# End-to-End Customer Intelligence System

This project builds a machine learning-based system to predict churn and recommend business actions.

In [417]:
import pandas as pd
import numpy as np

In [418]:
df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [419]:
df['Churn'] = df['Churn'].map({'Yes':1, 'No':0})

df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

df.drop('customerID', axis=1, inplace=True)

df = pd.get_dummies(df, drop_first=True)

print(df.dtypes)
print(df.shape)

SeniorCitizen                              int64
tenure                                     int64
MonthlyCharges                           float64
TotalCharges                             float64
Churn                                      int64
gender_Male                                 bool
Partner_Yes                                 bool
Dependents_Yes                              bool
PhoneService_Yes                            bool
MultipleLines_No phone service              bool
MultipleLines_Yes                           bool
InternetService_Fiber optic                 bool
InternetService_No                          bool
OnlineSecurity_No internet service          bool
OnlineSecurity_Yes                          bool
OnlineBackup_No internet service            bool
OnlineBackup_Yes                            bool
DeviceProtection_No internet service        bool
DeviceProtection_Yes                        bool
TechSupport_No internet service             bool
TechSupport_Yes     

In [420]:
from sklearn.model_selection import train_test_split

X = df.drop('Churn', axis=1)
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [421]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

In [422]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:,1]

In [423]:
def segment(prob):
    if prob > 0.7:
        return "High Risk"
    elif prob > 0.4:
        return "Medium Risk"
    else:
        return "Low Risk"

In [424]:
df['Churn_Probability'] = model.predict_proba(X)[:,1]
df['Risk_Segment'] = df['Churn_Probability'].apply(segment)

In [425]:
df[['Churn_Probability','Risk_Segment']].head()

,Churn_Probability,Risk_Segment
0,0.29,Low Risk
1,0.04,Low Risk
2,0.73,High Risk
3,0.00,Low Risk
4,0.80,High Risk


In [426]:
# @title
def action(seg):
    if seg == "High Risk":
        return "Give Discount"
    elif seg == "Medium Risk":
        return "Send Promotion"
    else:
        return "Upsell"

df['Recommended_Action'] = df['Risk_Segment'].apply(action)

In [427]:
df[['Risk_Segment','Recommended_Action']].head()

,Risk_Segment,Recommended_Action
0,Low Risk,Upsell
1,Low Risk,Upsell
2,High Risk,Give Discount
3,Low Risk,Upsell
4,High Risk,Give Discount


## Business Insights

- High risk customers should receive targeted discounts  
- Medium risk customers can be engaged with promotions  
- Low risk customers are suitable for upselling strategies  

This system enables data-driven decision making for customer retention.

In [428]:
df = df.drop(columns=['Risk_Segment', 'Recommended_Action'], errors='ignore')

In [429]:
X = df.drop('Churn', axis=1)
y = df['Churn']

In [430]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [431]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

In [432]:
from sklearn.metrics import accuracy_score

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))

Accuracy: 0.7856635911994322


In [433]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.82      0.90      0.86      1036
           1       0.63      0.46      0.53       373

    accuracy                           0.79      1409
   macro avg       0.73      0.68      0.70      1409
weighted avg       0.77      0.79      0.77      1409



In [434]:
y_prob = model.predict_proba(X_test)[:,1]

In [435]:
y_prob[:10]

array([1.        , 0.01      , 0.        , 0.95      , 0.        ,
       0.01      , 0.        , 0.        , 0.02      , 0.10585714])

In [436]:
X_test = X_test.copy()
X_test['Churn_Probability'] = y_prob

In [437]:
X_test[['Churn_Probability']].head()

,Churn_Probability
185,1.00
2715,0.01
3825,0.00
1807,0.95
132,0.00


In [438]:
X_test['Risk_Segment'] = X_test['Churn_Probability'].apply(segment)

In [439]:
X_test[['Churn_Probability','Risk_Segment']].head(10)

,Churn_Probability,Risk_Segment
185,1.000000,High Risk
2715,0.010000,Low Risk
3825,0.000000,Low Risk
1807,0.950000,High Risk
132,0.000000,Low Risk
1263,0.010000,Low Risk
3732,0.000000,Low Risk
1672,0.000000,Low Risk
811,0.020000,Low Risk
2526,0.105857,Low Risk


In [440]:
X_test['Risk_Segment'].value_counts()

,count
Risk_Segment,
Low Risk,1125
High Risk,245
Medium Risk,39


In [441]:
def action(seg):
    if seg == "High Risk":
        return "Offer Discount"
    elif seg == "Medium Risk":
        return "Send Promotion"
    else:
        return "Upsell Opportunity"

In [442]:
X_test['Recommended_Action'] = X_test['Risk_Segment'].apply(action)

In [443]:
X_test[['Churn_Probability','Risk_Segment','Recommended_Action']].head(10)

,Churn_Probability,Risk_Segment,Recommended_Action
185,1.000000,High Risk,Offer Discount
2715,0.010000,Low Risk,Upsell Opportunity
3825,0.000000,Low Risk,Upsell Opportunity
1807,0.950000,High Risk,Offer Discount
132,0.000000,Low Risk,Upsell Opportunity
1263,0.010000,Low Risk,Upsell Opportunity
3732,0.000000,Low Risk,Upsell Opportunity
1672,0.000000,Low Risk,Upsell Opportunity
811,0.020000,Low Risk,Upsell Opportunity
2526,0.105857,Low Risk,Upsell Opportunity


In [444]:
model.fit(X_train, y_train)

df['Churn_Probability'] = model.predict_proba(X)[:,1]

def segment(prob):
    if prob > 0.7:
        return "High Risk"
    elif prob > 0.4:
        return "Medium Risk"
    else:
        return "Low Risk"

df['Risk_Segment'] = df['Churn_Probability'].apply(segment)

df.to_csv("customer_dashboard_data.csv", index=False)

## Business Conclusion

This system enables companies to identify high-risk customers and take proactive actions such as offering discounts or targeted promotions.

By combining machine learning with business rules, organizations can improve customer retention and optimize revenue strategies.